In [6]:
%run "LangChain.ipynb"
%run "LangGraph.ipynb"
%run "DeepAgent.ipynb"


response='Your current plan is unlimited_plus with a monthly bill of $85. Changing to business_50gb will cost $35 more per month. Additionally, there is a network outage in your area with 5 hours of downtime and voice services are affected. A technician has been dispatched.' category='plan_change' summary='lookup_account for AC70345, request_plan_change to business_50gb, check_network_status for area code 718'
Paused at interrupt: [Interrupt(value={'action': 'plan_change', 'customer_id': 'AC80456', 'requested_plan': 'business_50gb', 'calculated_quote': {'available': True, 'current_plan': 'business_50gb', 'new_plan': 'business_50gb', 'price_diff_usd': 0}, 'question': 'Approve this plan change? (yes/no)'}, id='4aba69fb9d0e81108fe98888e0b304d8')]
--------------------------------------------------------
Final Answer:
[human] Check network status for account AC80456. Check my account, tell me if there is a network outageand I want to change my plan to business_50gb.
[ai] Network status for 

NameError: name 'stage3_agent' is not defined

NameError: name 'stage3_agent' is not defined

In [7]:
import time
import uuid

import gradio as gr
from langgraph.types import Command

STAGE_BADGE = {
    "Stage 1 — LangChain": "🔗 Single agent · direct tool calling",
    "Stage 2 — LangGraph": "🕸️ Supervisor graph · human approval required for plan changes",
    "Stage 3 — Deep Agent": "🧠 Deep agent · delegates to sub-agents",
}

EXAMPLE_PROMPTS = [
    "My account id is AC70345 — is my area having a network outage?",
    "Account AC50103 wants to change their plan to business_50gb.",
    "Check network status for AC80456 and quote a change to family_100gb.",
]


def new_state():
    return {"thread_id": f"gradio-{uuid.uuid4().hex[:8]}", "pending_interrupt": None}


def _message_text(message):
    """Normalize either a dict-style or LangChain message object to plain text."""
    if isinstance(message, dict):
        return message.get("content", "")
    return getattr(message, "content", str(message))

def ask_stage1(message: str) -> str:
    """Stage 1 — single LangChain agent with structured output."""
    result = stage1_agent.invoke({"messages": [{"role": "user", "content": message}]})
    structured = result.get("structured_response")
    if structured is not None:
        return structured.response
    return _message_text(result["messages"][-1])

def ask_stage2(message: str, state: dict):
    """Stage 2 — LangGraph supervisor graph. May pause on a human-approval interrupt."""
    config = {"configurable": {"thread_id": state["thread_id"]}}
    result = stage2_graph.invoke(
        {"messages": [{"role": "user", "content": message}], "handled": []},
        config=config,
    )

    interrupt_payload = result.get("__interrupt__")
    if interrupt_payload:
        state["pending_interrupt"] = interrupt_payload
        payload = interrupt_payload[0].value
        action = payload.get("action", "this action")
        quote = payload.get("calculated_quote") or payload.get("calculated_credit")

        reply = f"⏸️ **This needs approval before it can continue** ({action})."
        if quote:
                        if quote.get("eligible"):
                            reply += f"\n💰 Calculated credit: ${quote['credit_amount_usd']}"
                        elif quote.get("available"):
                            reply += f"\n📶 New plan: {quote.get('new_plan')}, monthly price change: ${quote.get('price_diff_usd', 0)}"
                        elif quote.get("eligible") is False:
                            reply += f"\nℹ️ Note: {quote.get('reason')}"
                        elif quote.get("available") is False:
                            reply += f"\nℹ️ Note: {quote.get('reason')}"
        reply += "\n\n👉 Use the Approve / Deny buttons below."
        return reply, state

    state["pending_interrupt"] = None
    last = result["messages"][-1]
    return _message_text(last), state


def resolve_stage2_interrupt(approve: bool, state: dict) -> str:
    config = {"configurable": {"thread_id": state["thread_id"]}}
    resume_value = "yes" if approve else "no"
    final_state = stage2_graph.invoke(Command(resume=resume_value), config=config)
    state["pending_interrupt"] = None
    last = final_state["messages"][-1]
    return _message_text(last)

def ask_stage3(message: str) -> str:
    """Stage 3 — deep agent that delegates to sub-agents via task().

    Delegation depends on the coordinator model supporting nested tool schemas
    (deepagents' internal task() tool). If that fails — e.g. a tool_use_failed
    error from a model that can't handle it — this degrades to a friendly
    message instead of crashing the UI.
    """
    try:
        result = stage3_agent.invoke({"messages": [{"role": "user", "content": message}]})
        structured = result.get("structured_response")
        if structured is not None:
            return structured.response
        return _message_text(result["messages"][-1])
    except Exception as e:
        return (
            f"⚠️ The Deep Agent couldn't complete sub-agent delegation ({e.__class__.__name__}). "
            f"Try Stage 1 or Stage 2 for this request, or check the Stage 3 coordinator model "
            f"in DeepAgent.ipynb — deepagents' task() tool needs a model that handles nested "
            f"tool schemas well (openai/gpt-oss-120b on Groq works; llama-3.3-70b-versatile "
            f"often doesn't)."
        )

def respond(message, chat_history, mode, state):
    """Generator so the UI can show a live typing indicator before the real reply lands."""
    if not message.strip():
        yield chat_history, state, gr.update(visible=False)
        return

    if state is None:
        state = new_state()

    if state.get("pending_interrupt"):
        chat_history.append({"role": "user", "content": message})
        chat_history.append({
            "role": "assistant",
            "content": "⚠️ There\'s a pending approval above — please use the Approve/Deny buttons first.",
        })
        yield chat_history, state, gr.update(visible=False)
        return

    chat_history.append({"role": "user", "content": message})
    chat_history.append({"role": "assistant", "content": "TeleAssist is typing..."})
    yield chat_history, state, gr.update(visible=False)
    time.sleep(0.3)

    try:
        if mode == "Stage 1 — LangChain":
            reply = ask_stage1(message)
            chat_history[-1] = {"role": "assistant", "content": reply}
            yield chat_history, state, gr.update(visible=False)

        elif mode == "Stage 2 — LangGraph":
            reply, state = ask_stage2(message, state)
            chat_history[-1] = {"role": "assistant", "content": reply}
            show_buttons = state.get("pending_interrupt") is not None
            yield chat_history, state, gr.update(visible=show_buttons)

        else:  # Stage 3 — Deep Agent
            reply = ask_stage3(message)
            chat_history[-1] = {"role": "assistant", "content": reply}
            yield chat_history, state, gr.update(visible=False)

    except NameError as e:
        chat_history[-1] = {
            "role": "assistant",
            "content": (
                f"⚠️ Agent object not found ({e}). Make sure the `%run` cell above executed "
                f"successfully so `stage1_agent`, `stage2_graph`, and `stage3_agent` all exist."
            ),
        }
        yield chat_history, state, gr.update(visible=False)

def on_approve(chat_history, state):
    chat_history.append({"role": "user", "content": "(approved)"})
    chat_history.append({"role": "assistant", "content": "Processing approval..."})
    yield chat_history, state, gr.update(visible=False)
    reply = resolve_stage2_interrupt(True, state)
    chat_history[-2] = {"role": "user", "content": "✅ Approved"}
    chat_history[-1] = {"role": "assistant", "content": reply}
    yield chat_history, state, gr.update(visible=False)


def on_deny(chat_history, state):
    chat_history.append({"role": "user", "content": "(denied)"})
    chat_history.append({"role": "assistant", "content": "Processing denial..."})
    yield chat_history, state, gr.update(visible=False)
    reply = resolve_stage2_interrupt(False, state)
    chat_history[-2] = {"role": "user", "content": "❌ Denied"}
    chat_history[-1] = {"role": "assistant", "content": reply}
    yield chat_history, state, gr.update(visible=False)


def on_clear():
    return [], new_state(), gr.update(visible=False)


def on_mode_change(mode):
    return gr.update(value=f"<div id=\'stage-badge\'>{STAGE_BADGE[mode]}</div>")


def use_example(example_text):
    return example_text

CUSTOM_CSS = """
:root {
    --ta-purple: #7c3aed;
    --ta-pink: #ec4899;
}

/* Header banner */
#header-block {
    background: linear-gradient(135deg, var(--ta-purple) 0%, #a855f7 50%, var(--ta-pink) 100%);
    border-radius: 16px;
    padding: 20px 24px;
    margin-bottom: 10px;
    box-shadow: 0 4px 18px rgba(124, 58, 237, 0.25);
}
#header-block h2, #header-block p { color: #ffffff !important; }

/* Stage badge — glows and lifts slightly on hover */
#stage-badge {
    display: inline-block;
    font-size: 0.85em;
    font-weight: 600;
    padding: 4px 14px;
    border-radius: 999px;
    background: rgba(255, 255, 255, 0.18);
    color: #fff;
    margin-top: 6px;
    transition: background 0.25s ease, transform 0.25s ease;
}
#stage-badge:hover {
    background: rgba(255, 255, 255, 0.32);
    transform: scale(1.04);
}

/* Chat window */
#chatbot {
    max-width: 1090px;
    margin: 0 auto;
    border-radius: 14px !important;
    box-shadow: 0 2px 14px rgba(0, 0, 0, 0.08);
    transition: box-shadow 0.25s ease;
}
#chatbot:hover { box-shadow: 0 4px 20px rgba(0, 0, 0, 0.12); }

/* Example prompt chips */
.chip-btn {
    font-size: 0.78em !important;
    border-radius: 999px !important;
    border: 1px solid rgba(124, 58, 237, 0.35) !important;
    transition: transform 0.15s ease, box-shadow 0.15s ease, background 0.15s ease !important;
}
.chip-btn:hover {
    transform: translateY(-2px);
    box-shadow: 0 4px 10px rgba(124, 58, 237, 0.25);
    background: rgba(124, 58, 237, 0.08) !important;
}

/* Send button */
#send-btn {
    background: linear-gradient(135deg, var(--ta-purple), var(--ta-pink)) !important;
    color: #fff !important;
    border: none !important;
    transition: transform 0.15s ease, opacity 0.15s ease !important;
}
#send-btn:hover { transform: translateY(-1px); opacity: 0.92; }
#send-btn:active { transform: translateY(0) scale(0.97); }

/* Clear button gets a playful little tilt */
#clear-btn { transition: transform 0.2s ease !important; }
#clear-btn:hover { transform: rotate(-8deg) scale(1.05); }

/* Message box focus ring */
textarea, input[type="text"] {
    transition: box-shadow 0.2s ease, border-color 0.2s ease !important;
}
textarea:focus, input[type="text"]:focus {
    box-shadow: 0 0 0 3px rgba(124, 58, 237, 0.25) !important;
    border-color: var(--ta-purple) !important;
}

/* Approval row gently pulses while it's waiting on you */
#approval-row {
    border-radius: 12px;
    padding: 6px;
    animation: pulse-border 1.8s ease-in-out infinite;
}
@keyframes pulse-border {
    0%, 100% { box-shadow: 0 0 0 0 rgba(124, 58, 237, 0.28); }
    50%      { box-shadow: 0 0 0 8px rgba(124, 58, 237, 0); }
}
#approve-btn {
    background: #16a34a !important;
    color: #fff !important;
    border: none !important;
    transition: transform 0.15s ease, box-shadow 0.15s ease !important;
}
#approve-btn:hover { transform: scale(1.05); box-shadow: 0 4px 12px rgba(22, 163, 74, 0.35); }
#deny-btn {
    background: #dc2626 !important;
    color: #fff !important;
    border: none !important;
    transition: transform 0.15s ease, box-shadow 0.15s ease !important;
}
#deny-btn:hover { transform: scale(1.05); box-shadow: 0 4px 12px rgba(220, 38, 38, 0.35); }

/* Radio mode selector — highlight the active pill */
.gradio-container input[type="radio"]:checked + label {
    color: var(--ta-purple) !important;
    font-weight: 700 !important;
}
"""

with gr.Blocks(title="TeleAssist — Telecom Customer Support", css=CUSTOM_CSS, theme=gr.themes.Soft(primary_hue="violet")) as demo:
    with gr.Column(elem_id="header-block"):
        gr.Markdown("## 📱 TeleAssist — Telecom Customer Support")
        gr.Markdown("Chat with the support desk across three agent architectures.")
        stage_badge = gr.HTML(f"<div id='stage-badge'>{STAGE_BADGE['Stage 1 — LangChain']}</div>")

    mode = gr.Radio(
        ["Stage 1 — LangChain", "Stage 2 — LangGraph", "Stage 3 — Deep Agent"],
        value="Stage 1 — LangChain",
        label="Agent to talk to",
    )
    mode.change(on_mode_change, inputs=mode, outputs=stage_badge)

    chatbot = gr.Chatbot(label="Conversation", height=420, elem_id="chatbot", avatar_images=(None, "🤖"))
    state = gr.State(new_state())

    with gr.Row(visible=False, elem_id="approval-row") as approval_row:
        approve_btn = gr.Button("✅ Approve", elem_id="approve-btn")
        deny_btn = gr.Button("❌ Deny", elem_id="deny-btn")

    gr.Markdown("**Try asking:**")
    with gr.Row():
        example_btns = [gr.Button(p, elem_classes="chip-btn") for p in EXAMPLE_PROMPTS]

    with gr.Row():
        msg = gr.Textbox(
            placeholder="Ask about account info, plan, network status, or a plan change...",
            label="Message",
            scale=4,
        )
        send_btn = gr.Button("Send", scale=1, elem_id="send-btn")
        clear_btn = gr.Button("🗑️ Clear", scale=1, elem_id="clear-btn")

    for btn in example_btns:
        btn.click(use_example, inputs=btn, outputs=msg)

    send_btn.click(respond, [msg, chatbot, mode, state], [chatbot, state, approval_row]).then(
        lambda: "", None, msg
    )
    msg.submit(respond, [msg, chatbot, mode, state], [chatbot, state, approval_row]).then(
        lambda: "", None, msg
    )

    approve_btn.click(on_approve, [chatbot, state], [chatbot, state, approval_row])
    deny_btn.click(on_deny, [chatbot, state], [chatbot, state, approval_row])
    clear_btn.click(on_clear, None, [chatbot, state, approval_row])

demo.launch()


C:\Users\PMLS\AppData\Local\Temp\ipykernel_43268\2677546518.py:294: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="TeleAssist — Telecom Customer Support", css=CUSTOM_CSS, theme=gr.themes.Soft(primary_hue="violet")) as demo:


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [19]:
import gradio as gr
print(gr.__version__)

6.22.0
